# 04 – Graph Features

This notebook turns the **homophily** found in the EDA into features from the **transaction graph**, in two steps: **present-neighbour graph features** that describe each transaction's neighbourhood, and a **two-stage (stacked) model** that adds the predicted risk of the neighbours. Each step is tested against the baseline (LightGBM with class weighting, `03_baseline_models.ipynb`) on the same folds.

## Goal
The EDA found that illicit transactions cluster together: a labeled neighbour of an illicit transaction is illicit 57% of the time, against 3% for a licit one.
The question is whether a model can exploit this at test time, when no test transaction has a labeled neighbour.

## Two kinds of graph features
| Step | Features | Idea |
|---|---|---|
| 1. Present-neighbour features | How many neighbours a transaction has and how many are "present"; the mean and max of the neighbours' (compressed) local features | Describes the graph around a transaction and *what its neighbours look like* |
| 2. Neighbours' predicted risk | The mean and max of a first model's illicit score over the neighbours | Uses homophily directly: a transaction next to risky transactions is more likely illicit |

Neither uses the label of a neighbour, so both can be computed for test transactions.

## How I judge them
On the 4 **late folds** from `02_validation_setup.ipynb` (mean AUC), plus the public leaderboard scores of the submissions made from each step.

## 0. Setup
The graph features live in `src/features.py`, and the graph models in `src/models.py`, so notebook 05 reuses exactly the same code.

**Helpers**
- `build_graph(edges, train, test)`: numbers every transaction and stores the graph as sparse adjacency matrices.
- `fit_local_pca(pool)`: standardises the 93 local features and fits a 10-component PCA with fixed signs (no labels used).
- `add_graph_features(graph, train_rows, pred_rows, pca)`: appends the 27 graph columns to both sets of rows, under one present rule.
- `add_neighbour_risk(graph, train_rows, pred_rows, seed, balanced, use_offset)`: stage 1; appends the 2 neighbours' risk columns.
- `two_stage_model(graph, train_rows, pred_rows, seeds)`: averages a model function's predictions over several seeds.
- `evaluate_graph_model(model, train, graph, folds)`: runs a model function on each fold, rebuilding every feature from that fold's training period.

In [1]:
import sys
sys.path.append("..")  # add the repo root so `src` can be imported from inside notebooks/

import pandas as pd
from sklearn.metrics import roc_auc_score

from src.data import load_train, load_test, load_edges, labeled, save_submission, FEATURE_COLS, LABEL_COL
from src.features import build_graph, fit_local_pca, add_graph_features, GRAPH_COLS, PRESENCE_COLS
from src.models import fit_lightgbm, add_neighbour_risk, two_stage_model, average_over_seeds, RISK_COLS
from src.validation import LATE_FOLDS, evaluate_graph_model, log_experiment

pd.set_option("display.width", 150)  # print wide tables on one line

## 1. Load the data and build the graph
The graph is built once from the edge list. Every transaction numbered, so that "sum over my neighbours" becomes a fast sparse-matrix product instead of a Python loop.

In [2]:
train, test, edges = load_train(), load_test(), load_edges()
lab = labeled(train).reset_index(drop=True)  # labeled training rows (time steps 1-35)
graph = build_graph(edges, train, test)      # every transaction in the edge list, train.csv or test.csv
print(f"Labeled rows: {len(lab):,} | test rows: {len(test):,} | edges: {len(edges):,} | transactions in the graph: {len(graph.nodes):,}")

Labeled rows: 31,235 | test rows: 15,329 | edges: 234,355 | transactions in the graph: 203,769


## 2. Which neighbours to use: the "present" rule
My first attempt (step 4 of the project) averaged features over **every** neighbour that has a row in `train.csv` or `test.csv`. It did not improve validation, and the EDA (`01_eda.ipynb`, section 5) showed why it is also unsafe:
a labeled training transaction finds all of its neighbours in `train.csv` (including unlabeled ones), while a test transaction finds only about half of its neighbours in `test.csv`.
`test.csv` behaves like "the labeled transactions of the test period".

So a neighbour counts as **present** only if it is
- a labeled row of `train.csv`, for training transactions, or
- a row of `test.csv`, for test transactions.

This one rule is applied to training and test rows in the same call (`add_graph_features`), so both see their neighbourhoods in the same way.

### Compressing the neighbours' features
The first 93 features describe the transaction itself (**local** features, see the EDA's correlation blocks). Averaging all 93 over the neighbours would add 186 strongly correlated columns.
Instead, PCA first compresses the standardised local features into **10 components**. PCA uses no labels, so it is fitted on all `train.csv` rows, labeled or not.

Each transaction then gets 27 graph columns:
- 7 **presence** features: in-degree, out-degree, present neighbours (in, out, total), the share of neighbours that are present, and the number of absent neighbours;
- the **mean** and **max** of the 10 components over its present neighbours (20 columns, empty when no neighbour is present).

In [3]:
pca = fit_local_pca(train)  # standardise the 93 local features, then PCA to 10 components (no labels used)
print(f"Variance of the 93 local features kept by 10 components: {pca[-1].explained_variance_ratio_.sum():.1%}")

lab_g, test_g = add_graph_features(graph, lab, test, pca)  # + the 27 graph columns, one rule for both
print(f"Graph columns added: {len(GRAPH_COLS)}")
print(f"Rows with no present neighbour: training {lab_g['nb_mean_pc0'].isna().mean():.1%} | test {test_g['nb_mean_pc0'].isna().mean():.1%}")

# Average presence features: with the present rule, training and test rows should look alike
pd.DataFrame({"training rows": lab_g[PRESENCE_COLS].mean(), "test rows": test_g[PRESENCE_COLS].mean()}).round(3)

Variance of the 93 local features kept by 10 components: 67.1%
Graph columns added: 27
Rows with no present neighbour: training 21.8% | test 25.4%


,training rows,test rows
in_degree,1.763,2.018
out_degree,1.102,1.224
n_in_present,0.765,0.830
n_out_present,0.765,0.830
n_nb_present,1.530,1.660
share_present,0.631,0.609
n_nb_absent,1.335,1.582


**Takeaway:** With the present rule, training and test rows see their neighbourhoods in much the same way. 21.8% of training rows and 25.4% of test rows have no present neighbour, and on average 63.1% vs 60.9% of a transaction's neighbours are present. 
Test transactions have slightly more neighbours in total (in-degree 2.018 vs 1.763), but the share the model can see is about the same.

## 3. Do the graph features help?
Each model is written as a small **model function** `model(train_rows, pred_rows, seed)`, which trains on `train_rows` and returns the probability of illicit for `pred_rows`.

**Steps in `evaluate_graph_model`, for each late fold:**
1. Split the rows at the fold's cut-off and hide the validation labels before any feature is built.
2. Fit the PCA on the `train.csv` rows of the fold's training period only.
3. Build the graph features with present = the fold's training rows + its validation rows (the counterpart of labeled training rows + `test.csv`).
4. Call the model function and compute the AUC against the hidden labels.

Everything is rebuilt inside each fold, exactly like the submission, so no fold sees information from its own validation period.

In [4]:
def baseline(train_rows, pred_rows, seed):
    """LightGBM (balanced) on the 165 raw features: the best model of notebook 03."""
    return fit_lightgbm(train_rows, FEATURE_COLS, seed, balanced=True).predict_proba(pred_rows[FEATURE_COLS])[:, 1]


def graph_model(train_rows, pred_rows, seed):
    """The same LightGBM with the 27 present-neighbour graph columns added."""
    cols = FEATURE_COLS + GRAPH_COLS
    return fit_lightgbm(train_rows, cols, seed, balanced=True).predict_proba(pred_rows[cols])[:, 1]


results = {
    "baseline (165 raw features)": evaluate_graph_model(baseline, train, graph, folds=LATE_FOLDS),
    "+ graph features": evaluate_graph_model(graph_model, train, graph, folds=LATE_FOLDS),
}
for name, r in results.items():
    print(f"{name:30s} late folds: {r['auc'].round(5).tolist()} | mean {r['auc'].mean():.5f}")

baseline (165 raw features)    late folds: [0.99718, 0.99785, 0.99742, 0.99857] | mean 0.99776
+ graph features               late folds: [0.99773, 0.99831, 0.99796, 0.99879] | mean 0.99820


**Takeaway:** The graph features raise the AUC on **every** late fold, and the mean from 0.99776 to 0.99820 (+0.00044). The gain looks small because the folds are close to 1.0, but it is consistent across all four folds.

## 4. Neighbours' predicted risk (two stages)
Homophily says that a transaction next to illicit ones is likely to be illicit. Test transactions have no labeled neighbours, but they do have neighbours that a model can **score**.
The model therefore works in two stages:
1. **Stage 1:** a LightGBM on the 165 raw features scores every transaction. The mean and max score over each transaction's present neighbours become 2 new features (`nb_mean_risk`, `nb_max_risk`).
2. **Stage 2:** a second LightGBM is trained on the raw features + 27 graph columns + these 2 risk features (194 columns).

**Out-of-fold scores, so training rows look like test rows.** A test transaction's neighbours are scored by a model that never saw their labels, so a training transaction's neighbours must be too.
Their stage-1 scores come from 5 models, each trained without one group of time steps and scoring only that group (out-of-fold). Present neighbours always share a time step, so no neighbour is scored by a model that saw its own label.

**Worked example.** If a transaction's two present neighbours get stage-1 scores of -4.0 and 1.0 (log-odds), it receives `nb_mean_risk` = -1.5 and `nb_max_risk` = 1.0.

First, how well does each risk feature separate the classes on its own (training rows with a present neighbour, out-of-fold scores)?

In [5]:
lab_r, _ = add_neighbour_risk(graph, lab_g, test_g, seed=42, balanced=True, use_offset=False)
has_neighbour = lab_r["nb_max_risk"].notna()
print(f"Training rows with a present neighbour: {has_neighbour.mean():.1%}")
for col in RISK_COLS:
    print(f"Single-feature AUC of {col}: {roc_auc_score(lab_r.loc[has_neighbour, LABEL_COL], lab_r.loc[has_neighbour, col]):.3f}")

Training rows with a present neighbour: 78.2%
Single-feature AUC of nb_mean_risk: 0.839
Single-feature AUC of nb_max_risk: 0.811


**Takeaway:** On its own, the neighbours' mean risk separates illicit from licit with an AUC of 0.839 (max: 0.811), on the 78.2% of training rows that have a present neighbour. The other rows get an empty value, which LightGBM treats as missing.

Next, the full two-stage model on the late folds. Each fold trains 7 LightGBM models (5 out of fold, 1 on all of the fold's training rows, and stage 2), so this cell takes a few minutes.

In [6]:
def neighbour_risk_model(train_rows, pred_rows, seed):
    """Two stages: stage-1 scores -> neighbours' risk -> stage-2 LightGBM (balanced, like the baseline)."""
    return two_stage_model(graph, train_rows, pred_rows, seed, balanced=True, use_offset=False)


name = "+ neighbours' risk"
results[name] = evaluate_graph_model(neighbour_risk_model, train, graph, folds=LATE_FOLDS)
print(f"{name:30s} late folds: {results[name]['auc'].round(5).tolist()} | mean {results[name]['auc'].mean():.5f}")

+ neighbours' risk             late folds: [0.99825, 0.99886, 0.99857, 0.99904] | mean 0.99868


## 5. Results and leaderboard
One row per model: the AUC on each late fold, the mean, and the public leaderboard score of the submission made from it.
Every result is also logged in `experiments.csv`.

In [7]:
LEADERBOARD = {"baseline (165 raw features)": 0.94637, "+ graph features": 0.95121, "+ neighbours' risk": 0.95846}  # public scores

table = pd.DataFrame({name: r.set_index("valid_steps")["auc"] for name, r in results.items()}).T
table["mean AUC"] = table.mean(axis=1)
table["public leaderboard"] = pd.Series(LEADERBOARD)

for name, r in results.items():
    log_experiment(f"LightGBM (balanced) {name}", r, notes="04_graph_features, seed 42")
table.round(5)

valid_steps,25-35,27-35,29-35,31-35,mean AUC,public leaderboard
baseline (165 raw features),0.99718,0.99785,0.99742,0.99857,0.99776,0.94637
+ graph features,0.99773,0.99831,0.99796,0.99879,0.99820,0.95121
+ neighbours' risk,0.99825,0.99886,0.99857,0.99904,0.99868,0.95846


**Takeaway:** Each step raises the AUC on every late fold: +0.00044 for the graph features, then +0.00048 for the neighbours' risk. The public leaderboard moved the same way, by much more: 0.94637 → 0.95121 (+0.0048) → 0.95846 (+0.0073). 
The late folds pointed in the right direction both times, even though they understate the size of the gain.

## 6. Submissions
Both submissions average **3 seeds** (42, 43, 44): each seed changes LightGBM's random row and feature sampling, and averaging makes the result less dependent on one random draw.
The final training uses all labeled rows (steps 1-35); the present set is the labeled training rows plus the `test.csv` rows.

In [8]:
SEEDS = [42, 43, 44]

graph_probs = average_over_seeds(graph_model, lab_g, test_g, SEEDS)
print("Saved:", save_submission(test, graph_probs, "04_graph_features.csv").name)

risk_probs = average_over_seeds(neighbour_risk_model, lab_g, test_g, SEEDS)
print("Saved:", save_submission(test, risk_probs, "04_neighbour_risk.csv").name)

Saved: 04_graph_features.csv
Saved: 04_neighbour_risk.csv


These are the files behind the leaderboard scores in section 5: `04_graph_features.csv` was submitted as `probe_lgb_neighbour_features.csv` (0.95121) and `04_neighbour_risk.csv` as `probe_neighbour_risk.csv` (0.95846).

## 7. Summary
| Finding | Decision |
|---|---|
| A test transaction sees only part of its neighbours, unlike a labeled training transaction | Use only **present** neighbours (labeled training rows, test.csv rows), one rule for both |
| Present-neighbour graph features: +0.00044 on the late folds, leaderboard 0.94637 → 0.95121 | Keep the 27 present-neighbour graph features |
| Neighbours' predicted risk (two stages, out-of-fold scores): +0.00048 more on the late folds, leaderboard 0.95846 | Keep the two-stage model; it is the base of the final model |

**Next:** `05_model_comparison.ipynb` starts from the two-stage model, tests the last changes (class weighting and a per-step offset), compares all submissions and picks the final two.